In [1]:
# Install scikit-learn for machine learning and model evaluation
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import libraries for data preparation and model splitting
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load the prepared concrete dataset
df = pd.read_csv("../data/concrete_prepared.csv")

# Select the eight original variables used to predict strength
# We will evaluate the additional engineered features later
features = [
    "cement",
    "slag",
    "fly_ash",
    "water",
    "superplasticizer",
    "coarse_aggregate",
    "fine_aggregate",
    "age_days"
]

# Define our predictors (X) and target variable (y)
X = df[features]
y = df["strength_mpa"]

# Define each concrete recipe using its seven ingredient quantities
# Age is excluded because the same recipe may be tested at different ages
recipe_columns = features[:-1]

# Assign the same group ID to observations with identical recipes
groups = df.groupby(recipe_columns, sort=False).ngroup()

# Reserve approximately 20% of unique recipes for final testing
# Keep all observations from each recipe in the same split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# Generate training and testing row indices
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Separate the predictors and target values
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Check that no identical recipe appears in both sets
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

# Display the results
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("Training recipes:", len(train_groups))
print("Testing recipes:", len(test_groups))

print("Recipes appearing in both sets:",
      len(train_groups.intersection(test_groups)))

Training observations: 799
Testing observations: 206
Training recipes: 342
Testing recipes: 86
Recipes appearing in both sets: 0


In [3]:
# Import the models we want to compare
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

# Import tools for cross-validation
from sklearn.model_selection import GroupKFold, cross_validate

# Define the four models
models = {
    "Baseline (Mean)": DummyRegressor(strategy="mean"),

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=250,
        l2_regularization=1.0,
        random_state=42
    )
}

# Use 5-fold cross-validation
# Recipes stay together so the same recipe does not appear
# in both training and validation within a fold
cv = GroupKFold(n_splits=5)

# Select the metrics used to evaluate each model
# MAE = average absolute prediction error in MPa
# RMSE = prediction error that penalizes large mistakes more heavily
# R² = how well the model explains differences in strength
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

# Store the results from each model
results = []

# Train and validate each model using the same folds
for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups.iloc[train_idx],
        cv=cv,
        scoring=scoring
    )

    # Convert negative error scores back to positive values
    # Lower MAE and RMSE indicate better predictions
    results.append({
        "Model": name,
        "MAE": -scores["test_MAE"].mean(),
        "RMSE": -scores["test_RMSE"].mean(),
        "R2": scores["test_R2"].mean(),
        "MAE_Std": scores["test_MAE"].std()
    })

# Turn the results into a comparison table
results_df = pd.DataFrame(results)

# Rank the models from lowest to highest prediction error
results_df = results_df.sort_values("MAE")

# Display the model comparison
display(results_df.round(3))

,Model,MAE,RMSE,R2,MAE_Std
3,Gradient Boosting,4.150,5.665,0.870,0.601
2,Random Forest,4.580,6.137,0.848,0.561
1,Linear Regression,8.230,10.334,0.571,0.578
0,Baseline (Mean),12.801,15.790,-0.002,0.629


In [4]:
# Import tools for finding better model settings
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor

# Define the settings we want to experiment with
# These control how quickly the model learns and how complex it becomes
param_grid = {
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_iter": [150, 250, 350],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [10, 20],
    "l2_regularization": [0.0, 1.0]
}

# Create the Gradient Boosting model
gb_model = HistGradientBoostingRegressor(
    random_state=42
)

# Test 20 different parameter combinations
# Use grouped cross-validation to keep recipes separated
# Select the model with the lowest average prediction error
search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=5),
    random_state=42,
    n_jobs=-1
)

# Find the best settings using training data only
search.fit(
    X_train,
    y_train,
    groups=groups.iloc[train_idx]
)

# Save the best model for later evaluation
best_model = search.best_estimator_

# Display the best settings and cross-validation score
print("Best parameters:")
print(search.best_params_)

print("\nBest cross-validation MAE:",
      round(-search.best_score_, 3), "MPa")

# Show the five best parameter combinations
tuning_results = pd.DataFrame(search.cv_results_)

tuning_results["MAE"] = (
    -tuning_results["mean_test_score"]
)

display(
    tuning_results[
        [
            "params",
            "MAE",
            "std_test_score"
        ]
    ]
    .sort_values("MAE")
    .head(5)
    .round(3)
)

Best parameters:
{'min_samples_leaf': 10, 'max_leaf_nodes': 15, 'max_iter': 350, 'learning_rate': 0.08, 'l2_regularization': 1.0}

Best cross-validation MAE: 3.899 MPa


,params,MAE,std_test_score
0,"{'min_samples_leaf': 10, 'max_leaf_nodes': 15,...",3.899,0.516
17,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.908,0.674
3,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.932,0.670
7,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.935,0.673
11,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.937,0.605


In [5]:
# Import tools for finding better model settings
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor

# Define the settings we want to experiment with
# These control how quickly the model learns and how complex it becomes
param_grid = {
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_iter": [150, 250, 350],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [10, 20],
    "l2_regularization": [0.0, 1.0]
}

# Create the Gradient Boosting model
gb_model = HistGradientBoostingRegressor(
    random_state=42
)

# Test 20 different parameter combinations
# Use grouped cross-validation to keep recipes separated
# Select the model with the lowest average prediction error
search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=5),
    random_state=42,
    n_jobs=-1
)

# Find the best settings using training data only
search.fit(
    X_train,
    y_train,
    groups=groups.iloc[train_idx]
)

# Save the best model for later evaluation
best_model = search.best_estimator_

# Display the best settings and cross-validation score
print("Best parameters:")
print(search.best_params_)

print("\nBest cross-validation MAE:",
      round(-search.best_score_, 3), "MPa")

# Show the five best parameter combinations
tuning_results = pd.DataFrame(search.cv_results_)

tuning_results["MAE"] = (
    -tuning_results["mean_test_score"]
)

display(
    tuning_results[
        [
            "params",
            "MAE",
            "std_test_score"
        ]
    ]
    .sort_values("MAE")
    .head(5)
    .round(3)
)

Best parameters:
{'min_samples_leaf': 10, 'max_leaf_nodes': 15, 'max_iter': 350, 'learning_rate': 0.08, 'l2_regularization': 1.0}

Best cross-validation MAE: 3.899 MPa


,params,MAE,std_test_score
0,"{'min_samples_leaf': 10, 'max_leaf_nodes': 15,...",3.899,0.516
17,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.908,0.674
3,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.932,0.670
7,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.935,0.673
11,"{'min_samples_leaf': 10, 'max_leaf_nodes': 31,...",3.937,0.605


In [6]:
# Import the metrics needed to evaluate our final model
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Use our tuned Gradient Boosting model to predict strength
# These testing recipes were not used to train or tune the model
test_predictions = best_model.predict(X_test)

# Calculate the average absolute prediction error
test_mae = mean_absolute_error(y_test, test_predictions)

# Calculate RMSE, which penalizes large prediction errors
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

# Calculate R² to measure how well the model explains strength variation
test_r2 = r2_score(y_test, test_predictions)

# Calculate prediction bias
# Positive values mean the model overpredicts strength on average
test_bias = np.mean(test_predictions - y_test)

# Display the final testing results
print("FINAL MODEL PERFORMANCE")
print(f"Test MAE: {test_mae:.3f} MPa")
print(f"Test RMSE: {test_rmse:.3f} MPa")
print(f"Test R²: {test_r2:.3f}")
print(f"Average prediction bias: {test_bias:.3f} MPa")


# Create a table comparing actual and predicted strength
test_results = pd.DataFrame({
    "actual_strength": y_test.to_numpy(),
    "predicted_strength": test_predictions
})

# Calculate the prediction error for each observation
test_results["error"] = (
    test_results["predicted_strength"] -
    test_results["actual_strength"]
)

# Check how often the model overestimates strength
print("\nPREDICTION ERRORS")
print(
    "Overestimated strength:",
    (test_results["error"] > 0).sum(),
    "out of",
    len(test_results),
    "observations"
)

# Evaluate predictions against our example 40 MPa requirement
target = 40

# Identify mixtures predicted to meet the target
predicted_pass = test_results["predicted_strength"] >= target

# Identify mixtures that actually met the target
actual_pass = test_results["actual_strength"] >= target

# Count cases where the model predicted a pass but the test failed
false_passes = (predicted_pass & ~actual_pass).sum()

print("\n40 MPa TARGET EVALUATION")
print("Predicted to meet target:", predicted_pass.sum())
print("Incorrectly predicted to meet target:", false_passes)

# Calculate the false-pass rate among predicted qualifying observations
if predicted_pass.sum() > 0:
    false_pass_rate = false_passes / predicted_pass.sum() * 100
    print(f"False-pass rate: {false_pass_rate:.2f}%")

# Display the 10 largest strength overpredictions
# These cases are especially important for risk assessment
display(
    test_results.sort_values(
        "error", ascending=False
    ).head(10).round(2)
)

FINAL MODEL PERFORMANCE
Test MAE: 4.118 MPa
Test RMSE: 6.184 MPa
Test R²: 0.881
Average prediction bias: -0.315 MPa

PREDICTION ERRORS
Overestimated strength: 115 out of 206 observations

40 MPa TARGET EVALUATION
Predicted to meet target: 82
Incorrectly predicted to meet target: 11
False-pass rate: 13.41%


,actual_strength,predicted_strength,error
80,37.27,53.66,16.39
40,56.50,72.21,15.72
33,54.90,69.14,14.25
79,30.22,44.16,13.94
87,57.23,67.94,10.71
26,52.20,62.87,10.67
35,64.90,75.55,10.65
81,46.23,56.69,10.46
25,56.40,66.47,10.07
67,7.40,17.31,9.92


In [7]:
# Import tools for cross-validation and creating independent model copies
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

# Set up five-fold cross-validation
# Identical recipes stay together within each fold
cv = GroupKFold(n_splits=5)

# Store the training-set recipe groups
train_recipe_groups = groups.iloc[train_idx]

# Create an empty array to hold validation predictions
oof_predictions = np.empty(len(X_train))

# Train the selected model on four folds and predict the fifth
# Repeat until every training observation has a validation prediction
for train_fold, val_fold in cv.split(
    X_train, y_train, groups=train_recipe_groups
):

    # Create a fresh copy of our tuned model
    fold_model = clone(best_model)

    # Train without using the current validation fold
    fold_model.fit(
        X_train.iloc[train_fold],
        y_train.iloc[train_fold]
    )

    # Predict the observations held out in this fold
    oof_predictions[val_fold] = fold_model.predict(
        X_train.iloc[val_fold]
    )

# Calculate prediction errors
# Positive values mean predicted strength was higher than measured strength
overprediction_errors = (
    oof_predictions - y_train.to_numpy()
)

# Calculate the 95th percentile of overprediction errors
# This is an empirical screening buffer, not an engineering safety factor
prediction_buffer = max(
    0,
    np.quantile(overprediction_errors, 0.95)
)

# Calculate the proposed screening cutoff for our 40 MPa example
target_strength = 40
screening_cutoff = target_strength + prediction_buffer

# Display the results
print("Cross-validation MAE:",
      round(mean_absolute_error(y_train, oof_predictions), 3),
      "MPa")

print("95th-percentile overprediction buffer:",
      round(prediction_buffer, 2), "MPa")

print("Example strength requirement:",
      target_strength, "MPa")

print("Proposed prediction screening cutoff:",
      round(screening_cutoff, 2), "MPa")

Cross-validation MAE: 3.9 MPa
95th-percentile overprediction buffer: 8.02 MPa
Example strength requirement: 40 MPa
Proposed prediction screening cutoff: 48.02 MPa


In [8]:
# Focus on the test observations measured at 28 days
# Our business scenario requires 40 MPa at this curing age
test_28 = test_results.loc[
    X_test["age_days"].to_numpy() == 28
].copy()

# Identify which mixtures actually achieved 40 MPa
actual_pass = test_28["actual_strength"] >= 40

# Compare predictions with and without our screening buffer
screening_results = []

for name, cutoff in [
    ("Without buffer", 40),
    ("With buffer", screening_cutoff)
]:

    # Identify mixtures that pass the model's screening rule
    predicted_pass = test_28["predicted_strength"] >= cutoff

    # Count the mixtures selected for further testing
    selected = predicted_pass.sum()

    # Count mixtures incorrectly predicted to meet 40 MPa
    false_passes = (predicted_pass & ~actual_pass).sum()

    # Count mixtures correctly selected
    correct_passes = (predicted_pass & actual_pass).sum()

    # Calculate the percentage of selected mixtures that failed
    false_pass_rate = (
        false_passes / selected * 100
        if selected > 0 else None
    )

    # Save the results for comparison
    screening_results.append({
        "Approach": name,
        "Prediction cutoff": cutoff,
        "Selected": selected,
        "Correct passes": correct_passes,
        "False passes": false_passes,
        "False-pass rate (%)": false_pass_rate
    })

# Display the number of observations available for this evaluation
print("28-day test observations:", len(test_28))
print("Actually meeting 40 MPa:", actual_pass.sum())

# Compare the two screening approaches
display(pd.DataFrame(screening_results).round(2))

28-day test observations: 84
Actually meeting 40 MPa: 31


,Approach,Prediction cutoff,Selected,Correct passes,False passes,False-pass rate (%)
0,Without buffer,40.00,30,28,2,6.67
1,With buffer,48.02,18,18,0,0.00


In [9]:
# Import Joblib to save our trained machine learning model
from pathlib import Path
import joblib

# Find the models folder in our project
# Create it if it doesn't already exist
model_folder = Path("../models")
model_folder.mkdir(parents=True, exist_ok=True)

# Save the trained model along with the information
# our dashboard will need to make predictions
model_package = {
    "model": best_model,
    "features": features,
    "prediction_buffer_mpa": float(prediction_buffer),
    "example_target_mpa": 40.0
}

# Save everything in one file
# Streamlit will load this file when we build the dashboard
model_path = model_folder / "concrete_strength_model.joblib"

joblib.dump(model_package, model_path)

# Confirm that the file was created
print("Model saved:", model_path)
print("File exists:", model_path.exists())

Model saved: ..\models\concrete_strength_model.joblib
File exists: True


In [10]:
# Check which variables matter most to our model's predictions
from sklearn.inspection import permutation_importance
import pandas as pd

# Measure how much prediction accuracy drops when each variable is shuffled
# We use the test data to interpret the final model, not to change or retune it
importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Create a table showing each variable's importance
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance_MAE": importance.importances_mean,
    "Std": importance.importances_std
})

# Put the most important variables at the top
importance_df = importance_df.sort_values(
    "Importance_MAE", ascending=False
)

# Display the results
# Higher importance means shuffling that variable increased prediction error more
display(importance_df.round(3))

,Feature,Importance_MAE,Std
7,age_days,8.803,0.376
0,cement,6.728,0.496
3,water,2.928,0.291
1,slag,2.436,0.210
6,fine_aggregate,1.031,0.097
5,coarse_aggregate,1.000,0.145
4,superplasticizer,0.860,0.144
2,fly_ash,0.351,0.057


In [11]:
# STEP 3.2 — Identify concrete mixtures for management review

# Build a table of training recipes and their cross-validation predictions
# These predictions came from models that did not train on the
# corresponding recipe in each validation fold
candidate_data = X_train.copy()

# Add the actual laboratory strength and cross-validation prediction
candidate_data["actual_strength_mpa"] = y_train.to_numpy()
candidate_data["predicted_strength_mpa"] = oof_predictions

# Focus on our business requirement: concrete tested at 28 days
candidate_data = candidate_data[
    candidate_data["age_days"] == 28
].copy()

# Average repeated laboratory measurements for the same recipe
# This prevents recipes tested multiple times from appearing separately
candidate_data = (
    candidate_data.groupby(recipe_columns, as_index=False)
    .agg(
        actual_strength_mpa=("actual_strength_mpa", "mean"),
        predicted_strength_mpa=("predicted_strength_mpa", "mean"),
        test_count=("actual_strength_mpa", "size")
    )
)

# Calculate the total binder used in each recipe
candidate_data["total_binder"] = (
    candidate_data["cement"] +
    candidate_data["slag"] +
    candidate_data["fly_ash"]
)

# Calculate the water-to-binder ratio
candidate_data["water_binder_ratio"] = (
    candidate_data["water"] / candidate_data["total_binder"]
)

# Calculate how far the measured strength exceeds 40 MPa
candidate_data["strength_margin"] = (
    candidate_data["actual_strength_mpa"] - 40
)

# Shortlist recipes that meet both conditions:
# 1. Measured strength of at least 40 MPa
# 2. Cross-validation prediction of at least 48.02 MPa
# This is preliminary screening, not production approval
recommended_candidates = candidate_data[
    (candidate_data["actual_strength_mpa"] >= 40) &
    (candidate_data["predicted_strength_mpa"] >= screening_cutoff)
].copy()

# Rank qualifying recipes by total binder content
# Show stronger measured results first when binder content is equal
recommended_candidates = recommended_candidates.sort_values(
    ["total_binder", "actual_strength_mpa"],
    ascending=[True, False]
)

# Check how many recipes qualify
print("28-day training recipes:", len(candidate_data))
print("Recipes passing conservative screening:",
      len(recommended_candidates))

# Display the 10 qualifying recipes with the lowest binder content
display(
    recommended_candidates[
        [
            "cement", "slag", "fly_ash", "water",
            "superplasticizer", "coarse_aggregate",
            "fine_aggregate", "total_binder",
            "water_binder_ratio", "actual_strength_mpa",
            "predicted_strength_mpa", "strength_margin",
            "test_count"
        ]
    ].head(10).round(2)
)

28-day training recipes: 333
Recipes passing conservative screening: 54


,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,total_binder,water_binder_ratio,actual_strength_mpa,predicted_strength_mpa,strength_margin,test_count
133,214.90,53.80,121.89,155.63,9.61,1014.3,780.58,390.59,0.40,52.20,49.02,12.20,1
186,277.19,97.82,24.46,160.70,11.19,1061.7,782.46,399.47,0.40,63.14,52.64,23.14,1
306,405.00,0.00,0.00,175.00,0.00,1120.0,695.00,405.00,0.43,52.30,48.51,12.30,1
165,259.90,100.60,78.40,170.60,10.40,935.7,762.90,438.90,0.39,49.77,48.44,9.77,1
290,375.00,93.80,0.00,126.60,23.40,852.1,992.60,468.80,0.27,56.70,59.64,16.70,1
254,322.00,149.00,0.00,186.00,8.00,951.0,709.00,471.00,0.39,52.42,51.71,12.42,1
279,355.00,19.00,97.00,145.00,13.13,967.0,871.00,471.00,0.31,44.03,54.19,4.03,1
255,322.50,148.60,0.00,185.80,8.50,951.0,709.50,471.10,0.39,52.43,50.22,12.43,1
232,312.90,160.50,0.00,177.60,9.60,916.6,759.50,473.40,0.38,52.45,51.30,12.45,1
237,313.00,161.00,0.00,178.00,10.00,917.0,759.00,474.00,0.38,52.44,51.19,12.44,1


In [12]:
# Compare the 10 shortlisted recipes that use the least binder
# Include their actual and predicted strength so we can judge the trade-offs

top_candidates = recommended_candidates.head(10).copy()

# Calculate what percentage of the binder comes from slag and fly ash
top_candidates["replacement_pct"] = (
    (top_candidates["slag"] + top_candidates["fly_ash"]) /
    top_candidates["total_binder"]
) * 100

# Give each candidate an easy-to-reference ID
top_candidates.insert(
    0, "mix_id",
    [f"Mix {i:02d}" for i in range(1, len(top_candidates) + 1)]
)

# Show the key business metrics in a compact table
summary_columns = [
    "mix_id",
    "total_binder",
    "cement",
    "replacement_pct",
    "actual_strength_mpa",
    "predicted_strength_mpa",
    "strength_margin",
    "test_count"
]

print("TOP 10 CANDIDATES — LOWEST BINDER USE")
print(top_candidates[summary_columns].round(2).to_string(index=False))

# Save the full shortlist, including all ingredient quantities
# We'll use this file later for the dashboard
recommended_candidates.to_csv(
    "../data/concrete_candidate_shortlist.csv",
    index=False
)

TOP 10 CANDIDATES — LOWEST BINDER USE
mix_id  total_binder  cement  replacement_pct  actual_strength_mpa  predicted_strength_mpa  strength_margin  test_count
Mix 01        390.59  214.90            44.98                52.20                   49.02            12.20           1
Mix 02        399.47  277.19            30.61                63.14                   52.64            23.14           1
Mix 03        405.00  405.00             0.00                52.30                   48.51            12.30           1
Mix 04        438.90  259.90            40.78                49.77                   48.44             9.77           1
Mix 05        468.80  375.00            20.01                56.70                   59.64            16.70           1
Mix 06        471.00  322.00            31.63                52.42                   51.71            12.42           1
Mix 07        471.00  355.00            24.63                44.03                   54.19             4.03           1
Mi

In [13]:
# Create a copy of all 54 shortlisted recipes
comparison = recommended_candidates.copy()

# Calculate the percentage of binder made from slag and fly ash
comparison["replacement_pct"] = (
    (comparison["slag"] + comparison["fly_ash"]) /
    comparison["total_binder"]
) * 100

# Calculate how much the model overestimated or underestimated strength
# Positive values mean the prediction was higher than the actual result
comparison["prediction_error"] = (
    comparison["predicted_strength_mpa"] -
    comparison["actual_strength_mpa"]
)

# Give each recipe a reference ID using its existing row index
comparison["recipe_id"] = "R" + comparison.index.astype(str)

# Choose the information we need for management comparisons
columns = [
    "recipe_id",
    "total_binder",
    "cement",
    "replacement_pct",
    "predicted_strength_mpa",
    "actual_strength_mpa",
    "prediction_error",
    "test_count"
]

# OPTION 1: Find recipes using the least total binder
print("\nOPTION 1: LOWEST BINDER USE")

display(
    comparison.sort_values("total_binder")
    [columns].head(5).round(2)
)

# OPTION 2: Find recipes using the least cement
# This is a material-use comparison, not a verified cost ranking
print("\nOPTION 2: LOWEST CEMENT USE")

display(
    comparison.sort_values("cement")
    [columns].head(5).round(2)
)

# OPTION 3: Find recipes with the highest predicted strength
# These may provide more strength margin but require more material
print("\nOPTION 3: HIGHEST PREDICTED STRENGTH")

display(
    comparison.sort_values(
        "predicted_strength_mpa",
        ascending=False
    )[columns].head(5).round(2)
)


OPTION 1: LOWEST BINDER USE


,recipe_id,total_binder,cement,replacement_pct,predicted_strength_mpa,actual_strength_mpa,prediction_error,test_count
133,R133,390.59,214.90,44.98,49.02,52.20,-3.18,1
186,R186,399.47,277.19,30.61,52.64,63.14,-10.50,1
306,R306,405.00,405.00,0.00,48.51,52.30,-3.80,1
165,R165,438.90,259.90,40.78,48.44,49.77,-1.33,1
290,R290,468.80,375.00,20.01,59.64,56.70,2.94,1



OPTION 2: LOWEST CEMENT USE


,recipe_id,total_binder,cement,replacement_pct,predicted_strength_mpa,actual_strength_mpa,prediction_error,test_count
133,R133,390.59,214.90,44.98,49.02,52.20,-3.18,1
154,R154,525.00,250.00,52.38,51.61,67.87,-16.26,1
165,R165,438.90,259.90,40.78,48.44,49.77,-1.33,1
180,R180,575.00,275.00,52.17,57.66,76.24,-18.58,1
186,R186,399.47,277.19,30.61,52.64,63.14,-10.50,1



OPTION 3: HIGHEST PREDICTED STRENGTH


,recipe_id,total_binder,cement,replacement_pct,predicted_strength_mpa,actual_strength_mpa,prediction_error,test_count
322,R322,500.0,500.0,0.00,67.93,67.57,0.36,1
270,R270,526.9,337.9,35.87,67.40,49.90,17.51,1
307,R307,531.3,425.0,20.01,65.74,61.80,3.94,1
286,R286,553.0,366.0,33.82,65.53,65.91,-0.38,1
331,R331,540.0,540.0,0.00,64.95,61.89,3.06,1


In [14]:
# Select the four recipes we want to compare for management
selected_ids = ["R133", "R186", "R290", "R322"]

# Retrieve their information from our shortlisted recipes
selected_mixes = comparison[
    comparison["recipe_id"].isin(selected_ids)
].copy()

# Calculate how far each predicted strength exceeds our
# conservative screening cutoff of approximately 48.02 MPa
selected_mixes["screening_margin_mpa"] = (
    selected_mixes["predicted_strength_mpa"] - screening_cutoff
)

# Arrange the recipes in our preferred comparison order
selected_mixes = (
    selected_mixes.set_index("recipe_id")
    .loc[selected_ids]
    .reset_index()
)

# Select the ingredient quantities and performance measures
# needed to compare the complete concrete formulations
columns = [
    "recipe_id",
    "cement",
    "slag",
    "fly_ash",
    "water",
    "superplasticizer",
    "coarse_aggregate",
    "fine_aggregate",
    "total_binder",
    "replacement_pct",
    "predicted_strength_mpa",
    "actual_strength_mpa",
    "screening_margin_mpa",
    "test_count"
]

# Print the complete comparison without cutting off columns
print("CONCRETE MIXES FOR MANAGEMENT REVIEW")
print(selected_mixes[columns].round(2).to_string(index=False))

# Save the comparison for our Streamlit dashboard
selected_mixes.to_csv(
    "../data/concrete_mix_comparison.csv",
    index=False
)

CONCRETE MIXES FOR MANAGEMENT REVIEW
recipe_id  cement  slag  fly_ash  water  superplasticizer  coarse_aggregate  fine_aggregate  total_binder  replacement_pct  predicted_strength_mpa  actual_strength_mpa  screening_margin_mpa  test_count
     R133  214.90 53.80   121.89 155.63              9.61            1014.3          780.58        390.59            44.98                   49.02                52.20                  1.00           1
     R186  277.19 97.82    24.46 160.70             11.19            1061.7          782.46        399.47            30.61                   52.64                63.14                  4.62           1
     R290  375.00 93.80     0.00 126.60             23.40             852.1          992.60        468.80            20.01                   59.64                56.70                 11.62           1
     R322  500.00  0.00     0.00 140.00              4.00             966.0          853.00        500.00             0.00                   67.93         

In [15]:
# Create the final management comparison table
decision_table = selected_mixes.copy()

# Assign a purpose to each mixture
# These are preliminary recommendations for laboratory testing
decision_table["recommendation"] = decision_table["recipe_id"].map({
    "R133": "Material-efficiency alternative",
    "R186": "Priority laboratory trial",
    "R290": "Higher-strength alternative",
    "R322": "High-strength reference"
})

# Use R186 as the comparison point
# This allows us to show the material trade-offs clearly
baseline_binder = decision_table.loc[
    decision_table["recipe_id"] == "R186", "total_binder"
].iloc[0]

# Calculate the percentage difference in binder use
# Positive values indicate more binder than R186
decision_table["binder_difference_pct"] = (
    (decision_table["total_binder"] - baseline_binder)
    / baseline_binder
) * 100

# Calculate the predicted margin above our 40 MPa requirement
# This is a model estimate, not a guarantee of strength
decision_table["predicted_margin_mpa"] = (
    decision_table["predicted_strength_mpa"] - 40
)

# Set the order for our executive comparison
decision_table["priority"] = decision_table["recipe_id"].map({
    "R186": 1,
    "R133": 2,
    "R290": 3,
    "R322": 4
})

decision_table = decision_table.sort_values("priority")

# Display the information most relevant to management
display_columns = [
    "recipe_id",
    "recommendation",
    "total_binder",
    "binder_difference_pct",
    "cement",
    "predicted_strength_mpa",
    "predicted_margin_mpa",
    "screening_margin_mpa"
]

print("MANAGEMENT DECISION COMPARISON")

print(
    decision_table[display_columns]
    .round(2)
    .to_string(index=False)
)

# Save the complete decision table for the dashboard
decision_table.to_csv(
    "../data/concrete_management_decisions.csv",
    index=False
)

print("\nManagement decision table saved!")

MANAGEMENT DECISION COMPARISON
recipe_id                  recommendation  total_binder  binder_difference_pct  cement  predicted_strength_mpa  predicted_margin_mpa  screening_margin_mpa
     R186       Priority laboratory trial        399.47                   0.00  277.19                   52.64                 12.64                  4.62
     R133 Material-efficiency alternative        390.59                  -2.22  214.90                   49.02                  9.02                  1.00
     R290     Higher-strength alternative        468.80                  17.36  375.00                   59.64                 19.64                 11.62
     R322         High-strength reference        500.00                  25.17  500.00                   67.93                 27.93                 19.91

Management decision table saved!
